# Affordance generation demo

**Banana example:** given a crop and task *peel the banana*, run open-vocabulary affordance discovery (`affordance_gen.py`, discover mode). The VLM scores the eight base affordances and may propose a **novel** name such as `peelable`.

## Setup

```bash
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt

export OPENAI_API_KEY="your_api_key_here"
jupyter notebook affordance_generation.ipynb
```

`OPENAI_API_KEY` must be set in the environment before launching Jupyter (or set it in a local-only cell with `%env OPENAI_API_KEY=...` — just don't commit that cell). The next cell checks that it's present. Uses `VLM_MODEL = "gpt-5-mini"`.

In [ ]:
import os

# Do not commit API keys. Set in the shell before starting Jupyter, or use %env in a local-only cell.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = ""

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set for this kernel.")
else:
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Export it in your shell, use %env OPENAI_API_KEY=..., "
        "or launch Jupyter from a terminal that already has it."
    )

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time
from pathlib import Path

from IPython.display import Image, Markdown, display

def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "affordance_gen" / "affordance_gen.py").is_file():
            return p
    raise FileNotFoundError(
        "Could not find repo root (expected affordance_gen/affordance_gen.py)."
    )

REPO_ROOT = find_repo_root()
AFFORDANCE_GEN = REPO_ROOT / "affordance_gen"
if str(AFFORDANCE_GEN) not in sys.path:
    sys.path.insert(0, str(AFFORDANCE_GEN))

# Jupyter caches imports — reload after vocab.py edits (e.g. removing traversable).
for mod in ("vocab", "affordance_gen"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import affordance_gen  # noqa: E402
import vocab  # noqa: E402

_filter_redundant_novel_affordances = affordance_gen._filter_redundant_novel_affordances
merge_vlm_affordance_name = affordance_gen.merge_vlm_affordance_name
vlm_openai_discover_predict = affordance_gen.vlm_openai_discover_predict
BASE_VOCAB = vocab.BASE_VOCAB
BASE_VOCAB_SET = vocab.BASE_VOCAB_SET

assert "traversable" not in BASE_VOCAB_SET, (
    "Stale import: restart kernel and re-run, or check affordance_gen/vocab.py"
)

print(f"Repo root: {REPO_ROOT}")
print(f"vocab.py: {AFFORDANCE_GEN / 'vocab.py'}")
print(f"Base vocab ({len(BASE_VOCAB)}): {', '.join(BASE_VOCAB)}")
print(f"OPENAI_API_KEY set: {bool(os.environ.get('OPENAI_API_KEY'))}")


## Banana example


In [ ]:
TASK = "peel the banana"
IMAGE_PATH = REPO_ROOT / "publish_dataset" / "banana_003.jpg"
VLM_MODEL = "gpt-5-mini"

USE_EMBEDDINGS = False
MERGE_SIMILARITY_THRESHOLD = 0.82
REFERENCE_AFFORDANCES: set[str] = set()

image_path = Path(IMAGE_PATH).expanduser().resolve()
assert image_path.is_file(), f"Image not found: {image_path}"

print(f"Task: {TASK}")
print(f"Image: {image_path}")
print(f"Model: {VLM_MODEL}")


In [ ]:
display(Markdown(f"**Task:** {TASK}"))
display(Image(filename=str(image_path), width=400))


In [ ]:
def discover_affordances(
    image_path: Path,
    task: str,
    *,
    vlm_model: str,
    reference_affordances: set[str] | None = None,
    use_embeddings: bool = False,
    merge_threshold: float = 0.82,
) -> dict:
    """Run one discover-mode VLM call and return merged + filtered proposal set."""
    ref = set(reference_affordances or set())
    global_known = set(BASE_VOCAB) | ref

    mapper = None
    if use_embeddings:
        from affordance_gen import EmbeddingMapper

        mapper = EmbeddingMapper("sentence-transformers/all-MiniLM-L6-v2")

    latencies: list[float] = []
    raw_scores = vlm_openai_discover_predict(
        image_path,
        task,
        vlm_model,
        sorted(global_known),
        latency_sink=latencies,
    )

    merged_scores: dict[str, int] = {}
    for raw_k, val in raw_scores.items():
        canon = merge_vlm_affordance_name(
            raw_k, global_known, mapper, merge_threshold
        )
        v = int(val)
        merged_scores[canon] = max(merged_scores.get(canon, 0), v)

    active = sorted(k for k, v in merged_scores.items() if v == 1)
    affordances, novel_affordances = _filter_redundant_novel_affordances(active)

    return {
        "raw_scores": raw_scores,
        "merged_scores": merged_scores,
        "affordances": affordances,
        "base_affordances": [a for a in affordances if a in BASE_VOCAB_SET],
        "novel_affordances": novel_affordances,
        "latency_s": latencies[0] if latencies else None,
    }


In [ ]:
t0 = time.perf_counter()
result = discover_affordances(
    image_path,
    TASK,
    vlm_model=VLM_MODEL,
    reference_affordances=REFERENCE_AFFORDANCES,
    use_embeddings=USE_EMBEDDINGS,
    merge_threshold=MERGE_SIMILARITY_THRESHOLD,
)
elapsed = time.perf_counter() - t0

print(f"VLM latency: {result['latency_s']:.2f}s" if result["latency_s"] else "VLM latency: n/a")
print(f"Total cell time: {elapsed:.2f}s")
print()

novel = result["novel_affordances"]
base = result["base_affordances"]
if novel:
    display(Markdown(
        "### Novel affordances discovered\n"
        + ", ".join(f"`{n}`" for n in novel)
        + "\n\n*(not in the eight base names: "
        + ", ".join(f"`{b}`" for b in BASE_VOCAB)
        + ")*"
    ))
else:
    display(Markdown(
        "**No novel affordances this run** (base vocab only). "
        "Use `gpt-5-mini` and re-run once — batch discover often proposed `peelable` for this image."
    ))

print("Proposed affordances (active after merge + filter):")
print(f"  base ({len(base)}): {base or '(none)'}")
print(f"  novel ({len(novel)}): {novel or '(none)'}")
print(f"  all ({len(result['affordances'])}): {result['affordances'] or '(none)'}")


In [ ]:
def _scores_table(scores: dict[str, int], novel: list[str]) -> str:
    lines = ["| affordance | score | kind |", "|---|---:|---|"]
    for name in sorted(scores):
        active = scores[name] == 1
        mark = "**" if active else ""
        kind = "novel" if name in novel else ("base" if name in BASE_VOCAB_SET else "other")
        lines.append(f"| {mark}{name}{mark} | {scores[name]} | {kind} |")
    return "\n".join(lines)

display(Markdown("### Merged VLM scores (after canonicalization)"))
display(Markdown(_scores_table(result["merged_scores"], result["novel_affordances"])))

print("Raw VLM JSON keys (before merge):")
print(json.dumps(result["raw_scores"], indent=2, sort_keys=True))
